# WasteVision phase 1: rebuild the data and the splits
CPU runtime. Run all cells in order (Runtime > Run all). Every step also writes a log to `MyDrive/WasteVision_v3/logs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/WasteVision_v3/logs', exist_ok=True)
print('MOUNT OK')

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/WasteVision_v3/logs
exec > >(tee -a /content/drive/MyDrive/WasteVision_v3/logs/step2_code.log) 2>&1
echo "=== step2_code START $(date -u +%H:%M:%S) ==="
cd /content
if [ -d WasteVision/.git ]; then git -C WasteVision pull --ff-only; else git clone https://github.com/ayushdhasmana433-rgb/WasteVision; fi
cd WasteVision/pipeline
git -C /content/WasteVision log -1 --format='commit %h %s'
ls
pip install -q imagehash
echo "=== step2_code DONE $(date -u +%H:%M:%S) ==="


In [ ]:
import os
base = "/content/drive/MyDrive/WasteVision_final"
names = ["data/normalized_images.tar", "data/all_images_before_dedup.csv", "results/duplicate_pairs.csv.gz",
         "data/frozen_features_effv2b0.npz", "data/splits.csv"]
lines = []
for n in names:
    p = f"{base}/{n}"
    lines.append(f"{os.path.getsize(p)/1e6:10.1f} MB  {n}" if os.path.exists(p) else f"   MISSING     {n}")
print("\n".join(lines))
open("/content/drive/MyDrive/WasteVision_v3/logs/step3_inputs.log", "w").write("\n".join(lines) + "\n")
assert not any("MISSING" in l for l in lines), "input file missing"
print("INPUTS OK")

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/WasteVision_v3/logs
exec > >(tee -a /content/drive/MyDrive/WasteVision_v3/logs/step4_extract.log) 2>&1
echo "=== step4_extract START $(date -u +%H:%M:%S) ==="
mkdir -p /content/data
tar -xf /content/drive/MyDrive/WasteVision_final/data/normalized_images.tar -C /content/data
ls /content/data
echo "files: $(find /content/data -type f | wc -l)"
df -h /content | tail -1
echo "=== step4_extract DONE $(date -u +%H:%M:%S) ==="


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/WasteVision_v3/logs
exec > >(tee -a /content/drive/MyDrive/WasteVision_v3/logs/step5_extra.log) 2>&1
echo "=== step5_extra START $(date -u +%H:%M:%S) ==="
cd /content/WasteVision/pipeline
mkdir -p /content/drive/MyDrive/WasteVision_v3/data
python -u add_extra_images.py \
  --raw-dir /content/raw_extra \
  --data-root /content/data \
  --out-dir /content/drive/MyDrive/WasteVision_v3/data \
  --all-csv /content/drive/MyDrive/WasteVision_final/data/all_images_before_dedup.csv
echo "=== step5_extra DONE $(date -u +%H:%M:%S) ==="


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/WasteVision_v3/logs
exec > >(tee -a /content/drive/MyDrive/WasteVision_v3/logs/step6_splits.log) 2>&1
echo "=== step6_splits START $(date -u +%H:%M:%S) ==="
cd /content/WasteVision/pipeline
python -u rebuild_splits.py \
  --all-csv /content/drive/MyDrive/WasteVision_final/data/all_images_before_dedup.csv \
  --extra-csv /content/drive/MyDrive/WasteVision_v3/data/extra_images.csv \
  --features /content/drive/MyDrive/WasteVision_final/data/frozen_features_effv2b0.npz \
  --old-splits /content/drive/MyDrive/WasteVision_final/data/splits.csv \
  --images-root /content/data \
  --out-dir /content/drive/MyDrive/WasteVision_v3/data
echo "=== step6_splits DONE $(date -u +%H:%M:%S) ==="


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/WasteVision_v3/logs
exec > >(tee -a /content/drive/MyDrive/WasteVision_v3/logs/step7_audit.log) 2>&1
echo "=== step7_audit START $(date -u +%H:%M:%S) ==="
cd /content/WasteVision/pipeline
mkdir -p /content/drive/MyDrive/WasteVision_v3/audit
python -u make_audit_sheets.py \
  --splits /content/drive/MyDrive/WasteVision_v3/data/splits_v3.csv \
  --images-root /content/data \
  --out-dir /content/drive/MyDrive/WasteVision_v3/audit \
  --pairs /content/drive/MyDrive/WasteVision_final/results/duplicate_pairs.csv.gz \
  --all-csv /content/drive/MyDrive/WasteVision_final/data/all_images_before_dedup.csv \
  --skip-first 20 --max-pairs 180
ls -la /content/drive/MyDrive/WasteVision_v3/audit | head -40
echo "=== step7_audit DONE $(date -u +%H:%M:%S) ==="


In [ ]:
print(open("/content/drive/MyDrive/WasteVision_v3/data/count_table.md").read())
open("/content/drive/MyDrive/WasteVision_v3/logs/PHASE1_DONE.txt", "w").write("phase 1 finished\n")
print("PHASE 1 DONE")